# ITO5202 Assessment 1 — Analysing Historical Data with System Performance

**Student ID:** 29701201  **Unit:** ITO5202
**Dataset:** Brazilian E-Commerce Public Dataset by Olist ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce))

## Environment Setup 

Spark runs in local mode inside a Docker container, with the driver acting as the single executor. The configuration choices below are made deliberately because they affect every later measurement:

- `local[*]` uses every core the container exposes, so `defaultParallelism` equals the container's core count.
- `spark.driver.memory` must be set before the JVM starts, so this cell should be the first Spark call after a kernel restart.
- `spark.sql.session.timeZone = UTC`. The Olist timestamps are Brazilian local times with no zone attached. Parsing them in UTC stops the container's time zone (and any daylight-saving gaps) from silently shifting or nulling timestamps, which would otherwise change quarter boundaries and delivery delays.
- `spark.sql.shuffle.partitions` is reduced from the default 200 to 2 × cores. The full pipeline processes roughly 110k line items. 200 shuffle partitions would create mostly tiny tasks whose scheduling overhead exceeds their work. 

In [1]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os          # file paths and CPU count
import platform    # Python / OS version for the environment table

import pandas as pd  

# Spark configuration and entry points
from pyspark import SparkConf
from pyspark.sql import SparkSession

# DataFrame functions, window specifications and schema types
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

In [2]:
# ---------------------------------------------------------------
# Spark configuration
# ---------------------------------------------------------------

master = "local[*]"


app_name = "ITO5202_A1_Olist_29701201"

# Set up configuration parameters for Spark
spark_conf = (
    SparkConf()
    .setMaster(master)
    .setAppName(app_name)
    # Driver memory: in local mode the driver is also the executor, so this
    # is all the memory Spark has. It only takes effect when the JVM starts,
    # so restart the kernel before running this cell.
    .set("spark.driver.memory", "4g")
    # Parse timestamps exactly as written in the CSVs (no time-zone shifting)
    .set("spark.sql.session.timeZone", "UTC")
    # Hide console progress bars so the PDF export stays clean
    .set("spark.ui.showConsoleProgress", "false")
)

# ---------------------------------------------------------------
# SparkSession 
# ---------------------------------------------------------------
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # show errors only, to keep outputs readable

# ---------------------------------------------------------------
# Shuffle partitions
# ---------------------------------------------------------------
# After a shuffle (groupBy, join, window), data is split into this many
# partitions. 2 x cores gives each core about two tasks: enough to keep all
# cores busy without the overhead of the default 200 near-empty tasks.
SHUFFLE_PARTITIONS = sc.defaultParallelism * 2
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)

print(f"Spark {spark.version} session started: {app_name}")
print(f"Shuffle partitions set to {SHUFFLE_PARTITIONS} "
      f"({sc.defaultParallelism} cores x 2)")

Spark 4.1.1 session started: ITO5202_A1_Olist_29701201
Shuffle partitions set to 16 (8 cores x 2)


In [3]:
# ---------------------------------------------------------------
# Execution environment summary 
# ---------------------------------------------------------------
def container_memory_limit():
    """Return the memory limit Docker has placed on this container, if any.
    Checks cgroup v2 first, then cgroup v1."""
    for path in ("/sys/fs/cgroup/memory.max",
                 "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            raw = open(path).read().strip()
            if raw != "max" and int(raw) < 1 << 60:   # very large value = no limit
                return f"{int(raw) / 1024**3:.1f} GB"
            return "no limit set"
        except (OSError, ValueError):
            continue
    return "unknown"


env = {
    "Execution mode": f"local ({sc.master}), Docker container",
    "Spark version": spark.version,
    "Python version": platform.python_version(),
    "OS (container)": f"{platform.system()} {platform.release()}",
    "CPU cores visible to container": os.cpu_count(),
    "defaultParallelism": sc.defaultParallelism,
    "Driver memory": spark.conf.get("spark.driver.memory", "default (1g)"),
    "Container memory limit": container_memory_limit(),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    # AQE can re-optimise plans at runtime (e.g. coalescing partitions,
    # switching join strategy). It appears as AdaptiveSparkPlan in explain().
    "spark.sql.adaptive.enabled (AQE)": spark.conf.get("spark.sql.adaptive.enabled"),
    # Tables smaller than this are broadcast to every task instead of shuffled
    "spark.sql.autoBroadcastJoinThreshold": spark.conf.get("spark.sql.autoBroadcastJoinThreshold"),
    "Session time zone": spark.conf.get("spark.sql.session.timeZone"),
    "Spark Web UI": "http://localhost:4040",
}

pd.DataFrame(env.items(), columns=["Setting", "Value"])

,Setting,Value
0,Execution mode,"local (local[*]), Docker container"
1,Spark version,4.1.1
2,Python version,3.13.12
3,OS (container),Linux 7.0.12-linuxkit
4,CPU cores visible to container,8
5,defaultParallelism,8
6,Driver memory,4g
7,Container memory limit,no limit set
8,spark.sql.shuffle.partitions,16
9,spark.sql.adaptive.enabled (AQE),true


## Data Loading


Every table is loaded with a hand-written `StructType` schema rather than `inferSchema=True`, for three reasons:

1. Correctness: Inference would read `customer_zip_code_prefix` as an integer and drop leading zeros (e.g. `01310` becomes `1310`). It would also leave timestamps as strings if the format was not recognised.
2. Cost: Inference needs an extra full pass over each file before the real read, which would also distort the Part B timings.
3. Stable plans: Fixed types mean Catalyst produces the same logical plan on every run.

With an explicit schema, Spark maps CSV columns by position and ignores the header row. A schema written in the wrong column order would load silently but wrongly, so each file's header is checked against its schema before loading.

### Tables used
Six of the nine Olist files are needed:

| Table | Used for |
|---|---|
| `order_items` | Revenue (`price`) and `product_id`; the central fact table |
| `orders` | Order status, purchase and delivery timestamps |
| `customers` | Customer state |
| `products` | Product category |
| `category_translation` | English category names |
| `order_reviews` | Review scores, for the delivery-delay question |

`order_payments` is deliberately **not** used. Payments are recorded per order, not per item, so joining them to line items would duplicate rows and double-count revenue. `price` from `order_items` is the correct revenue measure. `sellers` and `geolocation` are not needed for the business question.

In [4]:
# ---------------------------------------------------------------
# File locations
# ---------------------------------------------------------------
# CSVs are stored in ./data/ next to this notebook (see data/README.md).
# The path is relative, so it works both on the host machine and inside the Docker container.
DATA_DIR = "data"

# Short table name -> CSV file name
FILES = {
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
}

# Error Message
missing = [f for f in FILES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing from ./{DATA_DIR}/: {missing}. See data/README.md.")
print("All data files found.")

All data files found.


In [5]:
# ---------------------------------------------------------------
# Explicit schemas
# ---------------------------------------------------------------
# Column ORDER must match each CSV exactly, because Spark maps columns by position when a schema is supplied.
# All fields are nullable: Spark's CSV reader treats every column as
# nullable regardless of the schema, so completeness of key columns is
# checked explicitly later.
# Note: Olist's own header misspells 'length' as 'lenght' in two product columns.
SCHEMAS = {
    "orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True),
    ]),
    "order_items": StructType([
        StructField("order_id", StringType(), True),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), True),
        StructField("seller_id", StringType(), True),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True),
    ]),
    "customers": StructType([
        StructField("customer_id", StringType(), True),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", StringType(), True),  # string keeps leading zeros
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), True),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", DoubleType(), True),
        StructField("product_length_cm", DoubleType(), True),
        StructField("product_height_cm", DoubleType(), True),
        StructField("product_width_cm", DoubleType(), True),
    ]),
    "category_translation": StructType([
        StructField("product_category_name", StringType(), True),
        StructField("product_category_name_english", StringType(), True),
    ]),
    "order_reviews": StructType([
        StructField("review_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("review_score", IntegerType(), True),
        StructField("review_comment_title", StringType(), True),
        StructField("review_comment_message", StringType(), True),
        StructField("review_creation_date", TimestampType(), True),
        StructField("review_answer_timestamp", TimestampType(), True),
    ]),
}

In [6]:
# ---------------------------------------------------------------
# Header validation and CSV loading
# ---------------------------------------------------------------
def check_header(name):
    """Compare the CSV header row with the schema's field names.
    Guards against a schema written in the wrong column order."""
    path = os.path.join(DATA_DIR, FILES[name])
    # Read only the first line of the file as plain text
    header = spark.read.text(path).limit(1).first()[0]
    # Remove quote marks and the invisible byte-order mark (\ufeff)
    # that appears at the start of the translation file
    file_cols = [c.strip().strip('"').lstrip("\ufeff") for c in header.split(",")]
    schema_cols = SCHEMAS[name].fieldNames()
    if file_cols != schema_cols:
        raise ValueError(f"{name}: header {file_cols} does not match schema {schema_cols}")


def load_csv(name, multiline=False):
    """Load one Olist CSV using its explicit schema."""
    check_header(name)
    reader = (
        spark.read
        .option("header", True)                                  # skip the header row
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")        # Olist timestamp format
        .option("mode", "PERMISSIVE")                            # bad values -> null 
    )
    if multiline:
        # Review comments contain line breaks and quote marks inside
        # quoted fields. Without these options each line break would be
        # read as a new row, corrupting the table.
        reader = reader.option("multiLine", True).option("escape", '"')
    return reader.schema(SCHEMAS[name]).csv(os.path.join(DATA_DIR, FILES[name]))


# Load each table.
# No data is read until an action (count, show, collect) is called.
orders_df               = load_csv("orders")
order_items_df          = load_csv("order_items")
customers_df            = load_csv("customers")
products_df             = load_csv("products")
category_translation_df = load_csv("category_translation")
order_reviews_df        = load_csv("order_reviews", multiline=True)

# Keep references in one place for the summary checks below
tables = {
    "orders": orders_df,
    "order_items": order_items_df,
    "customers": customers_df,
    "products": products_df,
    "category_translation": category_translation_df,
    "order_reviews": order_reviews_df,
}
print("Headers validated and all tables loaded.")

Headers validated and all tables loaded.


## Initial Exploration

In [7]:
# Confirm the schemas were applied (types, not all strings)
order_items_df.printSchema()
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [10]:
# ---------------------------------------------------------------
# Row counts
# ---------------------------------------------------------------
rows = [(name, df.count(),  len(df.columns)) for name, df in tables.items()]
pd.DataFrame(rows, columns=["Table", "Rows loaded", "Columns"])

,Table,Rows loaded,Columns
0,orders,99441,8
1,order_items,112650,7
2,customers,99441,5
3,products,32951,9
4,category_translation,71,2
5,order_reviews,99224,7


In [11]:
# ---------------------------------------------------------------
# Null counts per column 
# ---------------------------------------------------------------
def null_profile(name, df):
    # One pass over the table: for every column, sum 1 where the value is null (only columns with at least one null are shown)
    counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).first()
    return [(name, c, counts[c]) for c in df.columns if counts[c]]

profile = [r for name, df in tables.items() for r in null_profile(name, df)]
pd.DataFrame(profile, columns=["Table", "Column", "Null count"]) if profile else "No nulls found."

,Table,Column,Null count
0,orders,order_approved_at,160
1,orders,order_delivered_carrier_date,1783
2,orders,order_delivered_customer_date,2965
3,products,product_category_name,610
4,products,product_name_lenght,610
5,products,product_description_lenght,610
6,products,product_photos_qty,610
7,products,product_weight_g,2
8,products,product_length_cm,2
9,products,product_height_cm,2


In [12]:
# ---------------------------------------------------------------
# Order status distribution
# ---------------------------------------------------------------
# Note: delivery performance analysis done on only delivered orders.
n_orders = orders_df.count()

(orders_df
 .groupBy("order_status")
 .count()
 .withColumn("pct", F.round(100 * F.col("count") / n_orders, 2))
 .orderBy(F.desc("count"))
 .show())

+------------+-----+-----+
|order_status|count|  pct|
+------------+-----+-----+
|   delivered|96478|97.02|
|     shipped| 1107| 1.11|
|    canceled|  625| 0.63|
| unavailable|  609| 0.61|
|    invoiced|  314| 0.32|
|  processing|  301|  0.3|
|     created|    5| 0.01|
|    approved|    2|  0.0|
+------------+-----+-----+



In [13]:
# ---------------------------------------------------------------
# Orders per quarter and overall date range
# ---------------------------------------------------------------
# Shows whether the first and last quarters have enough orders
# to be ranked fairly for Part A.
(orders_df
 .withColumn("year", F.year("order_purchase_timestamp"))
 .withColumn("quarter", F.quarter("order_purchase_timestamp"))
 .groupBy("year", "quarter")
 .count()
 .orderBy("year", "quarter")
 .show(20))

orders_df.select(
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase")
).show()

+----+-------+-----+
|year|quarter|count|
+----+-------+-----+
|2016|      3|    4|
|2016|      4|  325|
|2017|      1| 5262|
|2017|      2| 9349|
|2017|      3|12642|
|2017|      4|17848|
|2018|      1|21208|
|2018|      2|19979|
|2018|      3|12820|
|2018|      4|    4|
+----+-------+-----+

+-------------------+-------------------+
|     first_purchase|      last_purchase|
+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|
+-------------------+-------------------+



In [14]:
# ---------------------------------------------------------------
# Customer distribution by state
# ---------------------------------------------------------------
# A low-cardinality dimension (27 states) that is naturally skewed
# towards São Paulo (SP). Useful for Part B.
n_customers = customers_df.count()

(customers_df
 .groupBy("customer_state")
 .count()
 .withColumn("pct", F.round(100 * F.col("count") / n_customers, 2))
 .orderBy(F.desc("count"))
 .show(27))

+--------------+-----+-----+
|customer_state|count|  pct|
+--------------+-----+-----+
|            SP|41746|41.98|
|            RJ|12852|12.92|
|            MG|11635| 11.7|
|            RS| 5466|  5.5|
|            PR| 5045| 5.07|
|            SC| 3637| 3.66|
|            BA| 3380|  3.4|
|            DF| 2140| 2.15|
|            ES| 2033| 2.04|
|            GO| 2020| 2.03|
|            PE| 1652| 1.66|
|            CE| 1336| 1.34|
|            PA|  975| 0.98|
|            MT|  907| 0.91|
|            MA|  747| 0.75|
|            MS|  715| 0.72|
|            PB|  536| 0.54|
|            PI|  495|  0.5|
|            RN|  485| 0.49|
|            AL|  413| 0.42|
|            SE|  350| 0.35|
|            TO|  280| 0.28|
|            RO|  253| 0.25|
|            AM|  148| 0.15|
|            AC|   81| 0.08|
|            AP|   68| 0.07|
|            RR|   46| 0.05|
+--------------+-----+-----+



In [15]:
# ---------------------------------------------------------------
# Data-quality checks
# ---------------------------------------------------------------
# Investigate how many reviews each order has.
reviews_per_order = order_reviews_df.groupBy("order_id").count()

# Investigate categories that exist in products but have
# NO match in the translation table
untranslated = (products_df
                .filter(F.col("product_category_name").isNotNull())
                .select("product_category_name").distinct()
                .join(category_translation_df, "product_category_name", "left_anti"))

# No. orders delivered 
delivered = orders_df.filter(F.col("order_status") == "delivered")

checks = [
    ("Orders with more than one review",
     reviews_per_order.filter("count > 1").count()),
    ("Duplicate review_id values",
     order_reviews_df.count() - order_reviews_df.select("review_id").distinct().count()),
    ("Products with no category",
     products_df.filter(F.col("product_category_name").isNull()).count()),
    ("Categories missing from translation table",
     untranslated.count()),
    ("Delivered orders missing a delivery date",
     delivered.filter(F.col("order_delivered_customer_date").isNull()).count()),
    ("Order items whose order_id is not in orders",
     order_items_df.join(orders_df, "order_id", "left_anti").count()),
]
pd.DataFrame(checks, columns=["Check", "Count"])

,Check,Count
0,Orders with more than one review,547
1,Duplicate review_id values,814
2,Products with no category,610
3,Categories missing from translation table,2
4,Delivered orders missing a delivery date,8
5,Order items whose order_id is not in orders,0


In [16]:
# Categories have no English name
print("Categories with no English translation:")
untranslated.show(truncate=False)

Categories with no English translation:
+---------------------------------------------+
|product_category_name                        |
+---------------------------------------------+
|pc_gamer                                     |
|portateis_cozinha_e_preparadores_de_alimentos|
+---------------------------------------------+



In [17]:
# ---------------------------------------------------------------
# Rows per key: Determine concentrated is each candidate key
# ---------------------------------------------------------------
def key_frequency_summary(df, key):
    freq = df.groupBy(key).count()          # rows per distinct key value
    total = df.count()
    n_keys = freq.count()
    top_1pct = max(1, n_keys // 100)        # number of keys in the top 1%
    top_share = (freq.orderBy(F.desc("count"))
                     .limit(top_1pct)
                     .agg(F.sum("count"))
                     .first()[0])
    # Exact percentiles of rows-per-key (0.0 = no approximation error)
    q = freq.approxQuantile("count", [0.5, 0.9, 0.99], 0.0)
    stats = freq.agg(F.max("count"), F.avg("count")).first()
    return {
        "Key": key,
        "Rows": total,
        "Distinct keys": n_keys,
        "Mean rows/key": round(stats[1], 2),
        "Median": q[0], "P90": q[1], "P99": q[2],
        "Max rows/key": stats[0],
        "Rows held by top 1% of keys (%)": round(100 * top_share / total, 2),
    }

pd.DataFrame([key_frequency_summary(order_items_df, "product_id"),
              key_frequency_summary(order_items_df, "order_id")]).set_index("Key").T

Key,product_id,order_id
Rows,112650.00,112650.00
Distinct keys,32951.00,98666.00
Mean rows/key,3.42,1.14
Median,1.00,1.00
P90,6.00,1.00
P99,33.00,3.00
Max rows/key,527.00,21.00
Rows held by top 1% of keys (%),22.09,4.31


In [18]:
# The ten best-selling products: the 'heavy' keys behind any skew
(order_items_df
 .groupBy("product_id")
 .count()
 .orderBy(F.desc("count"))
 .show(10, truncate=False))

+--------------------------------+-----+
|product_id                      |count|
+--------------------------------+-----+
|aca2eb7d00ea1a7b8ebd4e68314663af|527  |
|99a4788cb24856965c36a24e339b6058|488  |
|422879e10f46682990de24d770e7f83d|484  |
|389d119b48cf3043d311335e499d9c6b|392  |
|368c6c730842d78016ad823897a372db|388  |
|53759a2ecddad2bb87a079a1f1519f73|373  |
|d1c427060a0f73f6b889a5c7c61f2ac4|343  |
|53b36df67ebb7c41585e8d54d6772e08|323  |
|154e7e31ebfa092203795c972e5804a6|281  |
|3dd2a17168ec895c781a9191c1e95ad7|274  |
+--------------------------------+-----+
only showing top 10 rows
